In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

# ======================
# LOAD DATA
# ======================
train = pd.read_csv("/kaggle/input/titanic/train.csv")
test = pd.read_csv("/kaggle/input/titanic/test.csv")

test_ids = test["PassengerId"]
full = pd.concat([train, test], sort=False)

# ======================
# CLEAN FEATURE ENGINEERING
# ======================

# Fill missing
full["Fare"] = full["Fare"].fillna(full["Fare"].median())
full["Embarked"] = full["Embarked"].fillna(full["Embarked"].mode()[0])

# Extract Title
full["Title"] = full["Name"].str.extract(" ([A-Za-z]+).", expand=False)

full["Title"] = full["Title"].replace(
    ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'],
    "Rare"
)

full["Title"] = full["Title"].replace({
    "Mlle":"Miss",
    "Ms":"Miss",
    "Mme":"Mrs"
})

# Age by title median
full["Age"] = full.groupby("Title")["Age"].transform(
    lambda x: x.fillna(x.median())
)

# Family size
full["FamilySize"] = full["SibSp"] + full["Parch"] + 1

# IsAlone
full["IsAlone"] = (full["FamilySize"] == 1).astype(int)

# Drop noisy columns
full.drop(["PassengerId","Name","Ticket","Cabin"], axis=1, inplace=True)

# Dummies
full = pd.get_dummies(full, drop_first=True)
full = full.fillna(0)

# Split back
train_processed = full.iloc[:len(train)]
test_processed = full.iloc[len(train):]

X = train_processed.drop("Survived", axis=1)
y = train_processed["Survived"].astype(int)
X_test = test_processed.drop("Survived", axis=1)

# ======================
# STRONG BUT STABLE SVM
# ======================
model = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(
        C=5,
        gamma=0.02,
        kernel="rbf",
        probability=True,
        random_state=42
    ))
])

# CV check
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("CV Accuracy:", cross_val_score(model, X, y, cv=cv).mean())

# Train full model
model.fit(X, y)
pred = model.predict(X_test)

# Submission
submission = pd.DataFrame({
    "PassengerId": test_ids,
    "Survived": pred.astype(int)
})

submission.to_csv("submission.csv", index=False)

CV Accuracy: 0.8361308141359614
